# dbt Demo – ELT DuckDB-vel

Ebben a notebookban egy teljes dbt projektet hozunk létre és futtatunk **dbt-core** + **dbt-duckdb** segítségével, PostgreSQL nélkül.

**Amit megtanulunk:**
- dbt projekt inicializálás és konfiguráció
- Seed CSV-k betöltése
- Staging, intermediate és mart modellek írása
- `dbt run` és `dbt test` futtatás
- `dbt docs generate` és az eredmények DuckDB-vel lekérdezése

**Architektúra:**
```
CSV fájlok (seeds)
    │
    ▼
Staging modellek  (stg_customers, stg_products, stg_orders)
    │
    ▼
Mart modellek     (dim_customer, dim_product, fct_sales)
    │
    ▼
DuckDB lekérdezés (analitikus riportok)
```

## 0. Telepítés

`dbt-core`: a dbt alap csomag  
`dbt-duckdb`: DuckDB adapter (nincs szükség külső adatbázis szerverre)

### Technológia: dbt-core + dbt-duckdb telepítés

```
dbt-core: az ELT transzformációs framework (SQL + Jinja + DAG)
dbt-duckdb: a DuckDB adapter (dbt-core plugin)
```

**Miért dbt-duckdb és nem dbt-postgres?**
- Nincs szükség külső adatbázis szerverre – a DuckDB in-process fut
- Helyi fejlesztői környezetnek, oktatásnak tökéletes
- Produkciós migrációnál csak a `profiles.yml` változik (type: duckdb → snowflake/bigquery)


In [ ]:
!pip install dbt-core dbt-duckdb --quiet

### dbt projekt inicializálás – kézi struktúra-létrehozás

Normál esetben `dbt init project_name` interaktívan kérdezi a konfigurációt.
A notebookban kézzel hozzuk létre a struktúrát, hogy kontrollálható és reprodukálható legyen.

**A `run()` helper függvény:** subprocess-szel hívja a dbt CLI-t, és megjeleníti a kimenetet.
Ez szimulál egy terminal parancsot a notebookban.


In [ ]:
import subprocess, os, json, pathlib

# Munkamappa: a notebook könyvtárán belül hozzuk létre a dbt projektet
PROJECT_DIR = pathlib.Path("/home/jovyan/work/dbt_demo")
PROJECT_DIR.mkdir(exist_ok=True)

def run(cmd, cwd=PROJECT_DIR):
    """Shell parancs futtatása, kimenet megjelenítése."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=str(cwd))
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr[:2000])
    return result

print("Munkamappa:", PROJECT_DIR)

## 1. lépés – dbt projekt struktúra létrehozása

Kézzel hozzuk létre a szükséges könyvtárakat és konfigurációs fájlokat  
(a `dbt init` interaktív, ezért fájlokat írunk közvetlenül).

### dbt_project.yml – a projekt gerince

```yaml
models:
  dbt_demo:
    staging:
      +materialized: view    ← staging modellek VIEW-ként
    marts:
      +materialized: table   ← mart modellek TABLE-ként
```

**Materializáció döntés:**
- `view`: nem foglal tárhelyet, minden lekérdezésnél újraszámol – staging-hez ideális
- `table`: fizikailag tárolt – mart-okhoz ideális, mert a BI eszközök gyors lekérdezést várnak
- `incremental`: csak az új sorokat dolgozza fel – nagy táblákhoz (nem ebben a demóban)
- `ephemeral`: csak CTE-ként létezik – közbülső logikához, nem kell az adatbázisba

**A `+` prefix** jelenti, hogy ez az összes modellre vonatkozik a mappában,
nem csak egy konkrét modellre.


In [ ]:
# Könyvtár struktúra
for d in ["models/staging", "models/marts", "seeds", "tests", "macros"]:
    (PROJECT_DIR / d).mkdir(parents=True, exist_ok=True)

# dbt_project.yml
(PROJECT_DIR / "dbt_project.yml").write_text("""
name: 'dbt_demo'
version: '1.0.0'
config-version: 2
profile: 'dbt_demo'

model-paths: ['models']
seed-paths:  ['seeds']
test-paths:  ['tests']
macro-paths: ['macros']

models:
  dbt_demo:
    staging:
      +materialized: view
    marts:
      +materialized: table
""")

print("dbt_project.yml létrehozva")

### profiles.yml – az adatbázis-kapcsolat konfigurálása

```yaml
dbt_demo:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: /home/jovyan/work/dbt_demo/demo.duckdb
```

**Adatmérnöki döntés: miért NEM commitoljuk a profiles.yml-t Git-be?**
- Credentials (jelszavak, API kulcsok) nem kerülhetnek verziókezelőbe
- Különböző fejlesztőknek más connection string-jeik vannak
- Megoldás: `profiles.yml` a `~/.dbt/` mappában él (felhasználó home könyvtár), nem a projektben

**Több target (dev/prod/staging):**
```yaml
outputs:
  dev:  type: duckdb, path: dev.duckdb
  prod: type: snowflake, account: xyz, ...
```
`dbt run --target prod` váltja a target-et.


In [ ]:
# profiles.yml – DuckDB adapter konfigurálása
# A fájl a ~/.dbt/ mappába kerül (dbt alapértelmezett helye)
profiles_dir = pathlib.Path.home() / ".dbt"
profiles_dir.mkdir(exist_ok=True)

(profiles_dir / "profiles.yml").write_text("""
dbt_demo:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: /home/jovyan/work/dbt_demo/demo.duckdb
""")

print("profiles.yml létrehozva:", profiles_dir / "profiles.yml")

## 2. lépés – Seed CSV fájlok

A `seeds/` mappába kerülő CSV-k `dbt seed` paranccsal kerülnek az adatbázisba.  
Ez ideális kisméretű referencia adatokhoz (pl. termékkategóriák, régiók, tesztelési adatok).

### Seed CSV fájlok – referencia adatok betöltése

A `seeds/` mappában lévő CSV fájlok `dbt seed` paranccsal kerülnek az adatbázisba.

**Mire való a seed?**
- Kis méretű, ritkán változó referencia adatok (termékkategóriák, régiók, valutakódok)
- Tesztelési fixture adatok (mint ebben a demóban)
- Statikus mapping táblák (pl. postal code → régió)

**Mire NEM való a seed?**
- Nagy adatmennyiség (> néhány ezer sor) → inkább staging modell + forrás konfigurálás
- Rendszeresen változó adatok → inkább ELT pipeline

**Séma konfiguráció:**
A `seeds/properties.yml`-ben megadható az oszlopok típusa:
```yaml
seeds:
  - name: customers
    config:
      column_types:
        customer_id: integer
```


In [ ]:
import csv, io

customers_csv = """customer_id,name,email,segment,country
1,Kovács Péter,kovacs@example.com,Business,Hungary
2,Nagy Anna,nagy@example.com,Consumer,Hungary
3,Tóth Gábor,toth@example.com,Business,Romania
4,Szabó Éva,szabo@example.com,Consumer,Slovakia
5,Horváth Miklós,horvath@example.com,Business,Hungary
"""

products_csv = """product_id,name,category,price
101,Laptop Pro,Electronics,450000
102,Wireless Mouse,Electronics,8500
103,Office Chair,Furniture,65000
104,Standing Desk,Furniture,120000
105,USB-C Hub,Electronics,15000
"""

orders_csv = """order_id,customer_id,product_id,quantity,order_date,status
1001,1,101,1,2024-01-15,completed
1002,2,102,2,2024-01-16,completed
1003,3,103,1,2024-01-17,completed
1004,1,105,3,2024-01-18,completed
1005,4,104,1,2024-02-01,completed
1006,5,102,1,2024-02-03,completed
1007,2,101,1,2024-02-10,completed
1008,3,105,2,2024-02-15,completed
1009,1,103,1,2024-03-01,completed
1010,5,104,2,2024-03-05,completed
"""

(PROJECT_DIR / "seeds" / "customers.csv").write_text(customers_csv)
(PROJECT_DIR / "seeds" / "products.csv").write_text(products_csv)
(PROJECT_DIR / "seeds" / "orders.csv").write_text(orders_csv)

print("Seed fájlok létrehozva: customers.csv, products.csv, orders.csv")

### `dbt seed` futtatás – CSV → DuckDB tábla

A `dbt seed` parancs:
1. Beolvassa a CSV fájlokat a `seeds/` mappából
2. Létrehozza vagy frissíti a táblákat az adatbázisban
3. Megjeleníti: hány sor töltődött be melyik táblába

**Várható kimenet:**
```
Running with dbt=1.x.x
Found 3 seeds ...
  OK created seed file dbt_demo.customers [INSERT 5 in 0.05s]
  OK created seed file dbt_demo.products  [INSERT 5 in 0.04s]
  OK created seed file dbt_demo.orders    [INSERT 10 in 0.04s]
```


In [ ]:
# dbt seed futtatás
run("dbt seed")

## 3. lépés – Staging modellek

A staging modellek feladata:
- Oszlop elnevezési konvenciók egységesítése
- Típuskonverziók (pl. string → date)
- Egyszerű szűrések (pl. törölt rekordok kizárása)
- Forrástábla aliasok (`{{ source(...) }}`)

**Konvenció:** `stg_<source>__<entity>.sql` (kettős aláhúzás: forrás__entitás)

### Staging modellek – a dbt ELT réteg első szintje

**Naming convention:** `stg_<forrás>__<entitás>.sql` (kettős aláhúzás: forrás__entitás)

**Staging feladatok:**
- Átnevezés: `name AS customer_name` – konzisztens névhasználat
- Típuskonverzió: `CAST(order_date AS DATE)` – biztonságos típusok
- Szűrés: `WHERE status = 'completed'` – csak érvényes adatok

**`{{ ref('customers') }}` technológia:**
- A Jinja macro a `customers` seed-re hivatkozik
- dbt automatikusan meghatározza a helyes sémát és táblanevet
- A függőség bejegyzésre kerül a DAG-ba: `seeds → stg_orders`


In [ ]:
# stg_customers.sql
(PROJECT_DIR / "models" / "staging" / "stg_customers.sql").write_text("""
-- Staging: customers seed → typed, renamed
SELECT
    customer_id,
    name            AS customer_name,
    email,
    segment,
    country
FROM {{ ref('customers') }}
""")

# stg_products.sql
(PROJECT_DIR / "models" / "staging" / "stg_products.sql").write_text("""
-- Staging: products seed → typed, renamed
SELECT
    product_id,
    name        AS product_name,
    category,
    price       AS unit_price
FROM {{ ref('products') }}
""")

# stg_orders.sql
(PROJECT_DIR / "models" / "staging" / "stg_orders.sql").write_text("""
-- Staging: orders seed → typed, filter only completed
SELECT
    order_id,
    customer_id,
    product_id,
    quantity,
    CAST(order_date AS DATE) AS order_date,
    status
FROM {{ ref('orders') }}
WHERE status = 'completed'
""")

print("Staging modellek létrehozva: stg_customers, stg_products, stg_orders")

## 4. lépés – Mart modellek (dim + fact)

A mart modellek a Kimball-módszer szerinti dimenziótáblákat és ténytáblákat hozzák létre.
- `dim_customer`, `dim_product`: dimenziótáblák surrogate key-jel
- `fct_sales`: ténytábla, amely `ref()`-el hivatkozik a dimenziókra

**Fontos:** a `{{ ref('stg_orders') }}` gondoskodik arról, hogy a dbt először  
a staging modelleket futtassa, majd a mart modelleket.

### Mart modellek – dim + fact Kimball stílusban

**`{{ ref('stg_customers') }}`** – a staging modellre hivatkozás:
- dbt tudja, hogy `dim_customer` függ `stg_customers`-től
- Ezért `stg_customers` MINDIG előbb fut → helyes DAG sorrend garantált

**ROW_NUMBER() surrogate key generálás:**
```sql
ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_sk
```
Ez determinisztikus – ugyanazon adatokon mindig ugyanazt az SK-t adja.
Ha SCD2 szükséges, a dbt `snapshots` funkciót kell használni.

**fct_sales – a ref() hálózat csúcsa:**
```sql
FROM {{ ref('stg_orders') }}   AS o
JOIN {{ ref('dim_customer') }} AS c ON ...
JOIN {{ ref('dim_product') }}  AS p ON ...
```
A dbt ebből a DAG-ot építi: seeds → stg_* → dim_* → fct_sales


In [ ]:
# dim_customer.sql
(PROJECT_DIR / "models" / "marts" / "dim_customer.sql").write_text("""
-- Dimension: customers with surrogate key
SELECT
    ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_sk,
    customer_id                               AS customer_nk,
    customer_name,
    email,
    segment,
    country
FROM {{ ref('stg_customers') }}
""")

# dim_product.sql
(PROJECT_DIR / "models" / "marts" / "dim_product.sql").write_text("""
-- Dimension: products with surrogate key
SELECT
    ROW_NUMBER() OVER (ORDER BY product_id) AS product_sk,
    product_id                              AS product_nk,
    product_name,
    category,
    unit_price
FROM {{ ref('stg_products') }}
""")

# fct_sales.sql
(PROJECT_DIR / "models" / "marts" / "fct_sales.sql").write_text("""
-- Fact: sales with SK lookups
SELECT
    o.order_id,
    c.customer_sk,
    p.product_sk,
    o.order_date,
    o.quantity,
    p.unit_price,
    o.quantity * p.unit_price AS revenue
FROM {{ ref('stg_orders') }}   AS o
JOIN {{ ref('dim_customer') }} AS c ON o.customer_id = c.customer_nk
JOIN {{ ref('dim_product') }}  AS p ON o.product_id  = p.product_nk
""")

print("Mart modellek létrehozva: dim_customer, dim_product, fct_sales")

### `dbt run` – a DAG alapú futtatás

```
dbt run
```

**A dbt automatikusan:**
1. Felépíti a DAG-ot a `ref()` hivatkozásokból
2. Topologiai sorrendben futtatja a modelleket
3. Staging modellek (VIEW) → dim modellek (TABLE) → fact modellek (TABLE)

**Várható kimenet:**
```
Running 6 models...
  OK view  model dbt_demo.stg_customers   [CREATE VIEW in 0.05s]
  OK view  model dbt_demo.stg_products    [CREATE VIEW in 0.04s]
  OK view  model dbt_demo.stg_orders      [CREATE VIEW in 0.04s]
  OK table model dbt_demo.dim_customer    [CREATE TABLE as SELECT in 0.08s]
  OK table model dbt_demo.dim_product     [CREATE TABLE as SELECT in 0.07s]
  OK table model dbt_demo.fct_sales       [CREATE TABLE as SELECT in 0.09s]
```


In [ ]:
# dbt run – összes modell futtatása
# A dbt automatikusan meghatározza a helyes sorrendet a ref() hivatkozások alapján:
# seeds → stg_* → dim_* → fct_*
run("dbt run")

## 5. lépés – Tesztelés

A `schema.yml` fájl definiálja az oszlopszintű generic testeket:
- `not_null`: nincs NULL érték
- `unique`: nincs duplikált érték  
- `accepted_values`: csak engedélyezett értékek
- `relationships`: hivatkozott táblában létezik az érték (FK jellegű ellenőrzés)

### schema.yml – adatminőség kódként

A `schema.yml` definiálja a modellek dokumentációját és generic testjeit.

**Négy beépített generic test:**
| Test | Mit ellenőriz? |
|------|----------------|
| `not_null` | Nincs NULL érték az oszlopban |
| `unique` | Nincs duplikált érték (pl. customer_id egyedi) |
| `accepted_values` | Csak az engedélyezett értékek fordulnak elő |
| `relationships` | FK integritás: minden érték létezik a hivatkozott táblában |

**Adatminőség mint kód:** a tesztek Git-ben verziókövetett YAML fájlok.
CI/CD pipeline-ban `dbt test` megbuktatja a merge-et, ha bármely teszt hibát jelez.


In [ ]:
# schema.yml a staging modellekhez
(PROJECT_DIR / "models" / "staging" / "schema.yml").write_text("""
version: 2

models:
  - name: stg_customers
    description: "Ügyfelek staging modellje"
    columns:
      - name: customer_id
        tests:
          - not_null
          - unique
      - name: email
        tests:
          - not_null

  - name: stg_products
    description: "Termékek staging modellje"
    columns:
      - name: product_id
        tests:
          - not_null
          - unique

  - name: stg_orders
    description: "Rendelések staging modellje (csak completed státuszú)"
    columns:
      - name: order_id
        tests:
          - not_null
          - unique
      - name: status
        tests:
          - accepted_values:
              values: ['completed']
      - name: customer_id
        tests:
          - relationships:
              to: ref('stg_customers')
              field: customer_id
      - name: product_id
        tests:
          - relationships:
              to: ref('stg_products')
              field: product_id
""")

# schema.yml a mart modellekhez
(PROJECT_DIR / "models" / "marts" / "schema.yml").write_text("""
version: 2

models:
  - name: fct_sales
    description: "Értékesítési ténytábla"
    columns:
      - name: order_id
        tests:
          - not_null
          - unique
      - name: revenue
        tests:
          - not_null
""")

print("schema.yml fájlok létrehozva")

### `dbt test` – adatminőségi ellenőrzés futtatása

```
dbt test
```

**Várható kimenet sikeres esetben:**
```
Running 8 tests...
  PASS not_null_stg_customers_customer_id
  PASS unique_stg_customers_customer_id
  PASS relationships_stg_orders_customer_id__customer_id__ref_stg_customers_
  PASS accepted_values_stg_orders_status__completed
  ...
All 8 tests passed!
```

**Ha egy teszt megbukik:**
```
FAIL 1
  FAIL  not_null_fct_sales_revenue  [FAILED in 0.12s]
  Got 2 results, configured to fail if != 0
```
A hibás sorok query-je megtekinthető: `target/run/` mappában a test SQL fájlban.


In [ ]:
# dbt test – összes teszt futtatása
# A kimenet mutatja: PASS / FAIL és a tesztelendő modell/oszlop nevét
run("dbt test")

## 6. lépés – Dokumentáció generálás

`dbt docs generate` létrehozza:
- `catalog.json`: az összes modell és oszlop metaadatait
- `manifest.json`: a DAG struktúrát (ref() hivatkozások)

Ezekből a `dbt docs serve` egy interaktív webes felületet indít,  
ahol megtekinthető a lineage gráf és a dokumentáció.

### `dbt docs generate` – automatikus dokumentáció

```
dbt docs generate
```

Két fájlt hoz létre a `target/` mappában:

**catalog.json:** minden modell és oszlop metaadatai
- Táblanév, séma, sorok száma
- Oszlop neve, típusa, nullable
- Tesztek listája oszloponként

**manifest.json:** a teljes DAG
- Modell függőségek (`child_map`, `parent_map`)
- Jinja kód a model forrásával
- `dbt docs serve` ezt a JSON-t rendereli interaktív webes felületté


In [ ]:
# dbt docs generate
run("dbt docs generate")

# catalog.json tartalmának megtekintése (modellek listája)
catalog_path = PROJECT_DIR / "target" / "catalog.json"
if catalog_path.exists():
    catalog = json.loads(catalog_path.read_text())
    print("\n=== Katalógusban szereplő modellek ===")
    for node_id, node in catalog.get("nodes", {}).items():
        name = node.get("metadata", {}).get("name", node_id)
        schema = node.get("metadata", {}).get("schema", "")
        cols = len(node.get("columns", {}))
        print(f"  {schema}.{name}: {cols} oszlop")
else:
    print("catalog.json nem található – nézd meg a dbt docs generate kimenetét")

### manifest.json – a lineage gráf forrása

A `child_map` megmutatja, mely modellek függnek egymástól:
```json
"model.dbt_demo.stg_orders": ["model.dbt_demo.fct_sales"]
```
Ez azt jelenti: ha `stg_orders` megváltozik, `fct_sales`-t is újra kell futtatni.

**`dbt run --select stg_orders+`** – ez a selektor szintaxis futtatja
a `stg_orders` modellt ÉS az összes downstream függőjét (`+` a függő modellek).


In [ ]:
# manifest.json – DAG struktúra megtekintése (ref() kapcsolatok)
manifest_path = PROJECT_DIR / "target" / "manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print("=== DAG – modell függőségek (child_map) ===")
    child_map = manifest.get("child_map", {})
    for parent, children in child_map.items():
        if "model." in parent and children:
            short = parent.split(".")[-1]
            deps = [c.split(".")[-1] for c in children if "model." in c or "test." in c]
            print(f"  {short} → {deps}")

## 7. lépés – Eredmények lekérdezése DuckDB-vel

A dbt által materializált táblák közvetlenül lekérdezhetők DuckDB Python API-val.  
Ez demonstrálja, hogy a dbt output ugyanolyan SQL táblaként érhető el, mint bármely más tábla.

### DuckDB analitika a dbt output-on

```python
con = duckdb.connect("/home/jovyan/work/dbt_demo/demo.duckdb")
```

A dbt által materializált táblák ugyanolyan DuckDB táblák – közvetlenül lekérdezhetők.
Ez demonstrálja az ELT pipeline végeredményét: tiszta, tesztelt, dokumentált fact tábla.


In [ ]:
import duckdb
import pandas as pd

# Kapcsolódás a dbt által létrehozott DuckDB fájlhoz
con = duckdb.connect("/home/jovyan/work/dbt_demo/demo.duckdb")

# Sémák és táblák listázása
print("=== Elérhető táblák ===")
con.execute("SHOW ALL TABLES").df()

### Bevétel kategóriánként – az ELT végeredménye

Ez az a lekérdezés, amelyért az egész pipeline készült. A `fct_sales JOIN dim_product`
3 tábla JOIN elegendő – a category már a dim_product-ban van (denormalizált).

**Az ELT értéke:** ugyanez az elemzés az ETL nélkül 4+ tábla JOIN-t igényelne a nyers adatokon.


In [ ]:
# Analitikus lekérdezés 1: Bevétel kategóriánként
print("=== Bevétel termék-kategóriánként ===")
query = """
SELECT
    p.category,
    COUNT(f.order_id)   AS orders,
    SUM(f.quantity)     AS total_units,
    SUM(f.revenue)      AS total_revenue
FROM main.fct_sales     AS f
JOIN main.dim_product   AS p ON f.product_sk = p.product_sk
GROUP BY p.category
ORDER BY total_revenue DESC
"""
con.execute(query).df()

### Top ügyfelek – konformált dimenziók kombinálása

`dim_customer` (szegmens, ország) és `dim_product` (kategória) egyszerre lekérdezhető
egyazon `fct_sales` ténytáblából. Ez a Kimball Bus Matrix lényege:
konformált dimenziók lehetővé teszik a cross-process analitikát.


In [ ]:
# Analitikus lekérdezés 2: Top ügyfelek szegmens szerinti bontásban
print("=== Top ügyfelek szegmens szerint ===")
query = """
SELECT
    c.customer_name,
    c.segment,
    c.country,
    COUNT(f.order_id)   AS orders,
    SUM(f.revenue)      AS total_revenue
FROM main.fct_sales     AS f
JOIN main.dim_customer  AS c ON f.customer_sk = c.customer_sk
GROUP BY c.customer_name, c.segment, c.country
ORDER BY total_revenue DESC
"""
con.execute(query).df()

### Havi bevétel trend – idősor az fct_sales-ből

```sql
DATE_TRUNC('month', order_date) AS month
```

**Megjegyzés:** ebben a demóban nincs `dim_date` tábla – az `order_date` közvetlenül
a fact táblában van. Produkciós dbt projektben ajánlott egy `dim_date` tábla
(generálható `dbt-date-spine` package-szel), mert lehetővé teszi a gap-filling
(0 rendeléses hónapok megjelenítése is) és az előszámított dátum attribútumokat.


In [ ]:
# Analitikus lekérdezés 3: Havi bevétel trend
print("=== Havi bevétel trend ===")
query = """
SELECT
    DATE_TRUNC('month', order_date) AS month,
    COUNT(order_id)                  AS orders,
    SUM(revenue)                     AS monthly_revenue
FROM main.fct_sales
GROUP BY DATE_TRUNC('month', order_date)
ORDER BY month
"""
con.execute(query).df()

## Összefoglalás – mit tanultunk?

| Fogalom | Leírás |
|---------|--------|
| **ELT** | Nyers adat betöltve → transzformáció a warehouse-ban SQL-lel |
| **dbt projekt** | `dbt_project.yml` + `profiles.yml` + `models/` könyvtár |
| **Staging modell** | Nyers → typed, renamed, filtered (`stg_*.sql`) |
| **Mart modell** | Dim és fact táblák (`dim_*.sql`, `fct_*.sql`) |
| **ref()** | Modell hivatkozás → automatikus DAG és futtatási sorrend |
| **dbt seed** | CSV → tábla betöltés (referencia adatokhoz) |
| **dbt run** | Összes modell futtatása a helyes sorrendben |
| **dbt test** | Generic tests (not_null, unique, relationships) |
| **dbt docs** | Automatikus dokumentáció + lineage gráf |
| **dbt-duckdb** | Könnyű, szerver nélküli adapter lokális fejlesztéshez |

**Következő lépések:**
- Incremental modellek: `{{ config(materialized='incremental') }}` + `is_incremental()` feltétel
- dbt snapshots: SCD2 automatikus kezelés
- dbt macros: Jinja újrafelhasználható blokkok
- CI/CD: `dbt run && dbt test` a pipeline-ban